# 01 — Data Pull

**Goal:** by the end of this notebook you have (1) ~3–5K oncology trials in parquet, and (2) ~1K synthetic oncology patients with biomarker profiles in JSON.

**Time:** ~1 hour the first time (mostly waiting on the API).

**Outputs:**
- `data/raw/trials_oncology.parquet`
- `data/raw/patients_cohort.json`


In [ ]:
# Setup
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')

from src.data import (
    ClinicalTrialsClient,
    oncology_query,
    studies_to_parquet,
    generate_cohort,
    save_cohort,
)

## 1. Inspect what the oncology query will look for

Before pulling thousands of trials, sanity-check the query.

In [ ]:
query = oncology_query()
print('Advanced filter:', query.advanced)
print('Overall status:', query.overall_status)
print('Page size:', query.page_size)

## 2. Dry run — pull just one page to verify shape

Always do this before a full pull. Confirms the API is reachable and the response parser works.

In [ ]:
with ClinicalTrialsClient() as client:
    sample = client.fetch_studies(query, limit=5)

print(f'Fetched {len(sample)} trials in the sample.')
print()
first = sample[0]
ident = first['protocolSection']['identificationModule']
elig = first['protocolSection'].get('eligibilityModule', {})
print('NCT ID:', ident['nctId'])
print('Title:', ident.get('briefTitle', '')[:120])
print()
print('Eligibility text (first 800 chars):')
print((elig.get('eligibilityCriteria') or '')[:800])

## 3. Full pull — all matching recruiting oncology trials

In [ ]:
with ClinicalTrialsClient() as client:
    trials = client.fetch_studies(query)

print(f'Pulled {len(trials)} trials.')

out_path = ROOT / 'data' / 'raw' / 'trials_oncology.parquet'
studies_to_parquet(trials, out_path)

## 4. Sanity checks on the pulled data

Quick distributions to confirm the data is what we expect before going further.

In [ ]:
import pandas as pd

df = pd.read_parquet(ROOT / 'data' / 'raw' / 'trials_oncology.parquet')
print('Trials:', len(df))
print()
print('Phases (first listed):')
print(df['phases'].apply(lambda x: x[0] if len(x) else 'NA').value_counts().head(10))
print()
print('Top conditions (first listed):')
print(df['conditions'].apply(lambda x: x[0] if len(x) else 'NA').value_counts().head(15))
print()
print('Eligibility text length distribution (chars):')
print(df['eligibility_criteria'].str.len().describe())

## 5. Generate the synthetic patient cohort

1000 patients with biomarker profiles sampled from published prevalence rates.

Note: `oversample_rare=True` boosts rare cancers (cholangiocarcinoma, mesothelioma, sarcoma) to ~5% each so we have enough hard cases for evaluation. Document this in the writeup as a known limitation.

In [ ]:
cohort = generate_cohort(n=1000, seed=42, oversample_rare=True)
save_cohort(cohort, ROOT / 'data' / 'raw' / 'patients_cohort.json')

from collections import Counter
types = Counter(p.cancer_type for p in cohort)
print('Cancer-type distribution:')
for k, v in sorted(types.items(), key=lambda x: -x[1]):
    print(f'  {k:25s} {v:>4d}  ({v/len(cohort)*100:.1f}%)')

## 6. Quick exploration: what does an eligibility text actually look like?

Read 3 random ones to start building intuition for the failure-mode taxonomy. As you read, jot notes into `notes/eligibility_observations.md`.

In [ ]:
import random
rng = random.Random(0)
for trial in rng.sample(list(df.to_dict('records')), 3):
    print('=' * 80)
    print(trial['nct_id'], '—', trial['brief_title'])
    print('Conditions:', trial['conditions'])
    print()
    print(trial['eligibility_criteria'])
    print()

## Next steps (Week 2)

- [ ] Build the embedding pipeline: chunk by criterion, embed via `bge-small`, index in Chroma.
- [ ] Write `patient_to_query()` that turns a `BiomarkerProfile` into a retrieval query.
- [ ] Start gold-standard annotation: 50 patient–trial pairs hand-labeled with failure-mode tags.

→ continue in `notebooks/04_embedding_retrieval.ipynb`.